In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

In [ ]:
def load_and_clean_data(lithium_path, ev_path):
    """Load lithium prices and EV sales, then prepare dates for analysis."""
    df = pd.read_csv(lithium_path, names=['date', 'price_cny'])
    ev = pd.read_csv(ev_path)

    # The dataset stores dates as decimal years (e.g., 2021.5).
    # We convert them to standard datetime so both datasets align on time.
    df['date'] = pd.to_datetime(
        df['date'].astype(str).str[:4] + '-01-01'
    ) + pd.to_timedelta((df['date'] % 1) * 365.25, unit='D')

    # Drop the fractional time part (23:47:49...) so the table shows clean dates.
    df['date'] = df['date'].dt.normalize()

    # Round prices to whole CNY — the decimal part comes from
    # WebPlotDigitizer extraction and is meaningless noise.
    df['price_cny'] = df['price_cny'].round(0).astype(int)

    df['Year'] = df['date'].dt.year

    return df, ev


def merge_data(df, ev, regions):
    """Filter EV sales by region and merge with lithium prices by year."""
    # Keep only regions that actually exist in the dataset.
    # This avoids silent failures if OWID renames an entity.
    available = ev['Entity'].unique()
    valid_regions = [r for r in regions if r in available]

    ev_filtered = ev[ev['Entity'].isin(valid_regions)].copy()
    merged = pd.merge(df, ev_filtered, on='Year', how='inner')
    return merged

In [ ]:
def calculate_correlation(merged):
    """Calculate Pearson correlation between price and EV sales for each region."""
    results = {}
    for region in merged['Entity'].unique():
        region_data = merged[merged['Entity'] == region]
        corr = region_data['price_cny'].corr(region_data['Electric cars sold'])
        results[region] = round(float(corr), 2)
    return results


def calculate_seasonality(df):
    """Calculate average lithium price per calendar month."""
    df = df.copy()
    df['Month'] = df['date'].dt.month
    seasonality = df.groupby('Month')['price_cny'].mean().reset_index()

    # Convert month numbers to short names (Jan, Feb) for readability.
    seasonality['Month'] = pd.to_datetime(
        seasonality['Month'], format='%m'
    ).dt.strftime('%b')
    return seasonality


def train_forecast(df, months_ahead=6):
    """Fit a linear trend on the last 12 months and forecast the next N months."""
    df_sorted = df.sort_values('date').reset_index(drop=True)
    last_12 = df_sorted.tail(12).copy()

    # Use days since the start of the window as the regression feature.
    last_12['days'] = (last_12['date'] - last_12['date'].min()).dt.days

    coeffs = np.polyfit(last_12['days'], last_12['price_cny'], 1)
    poly = np.poly1d(coeffs)

    # Anchor the forecast to the last actual price, so the line starts
    # where the historical data ends instead of jumping to the regression value.
    last_actual_price = last_12['price_cny'].iloc[-1]
    last_predicted_price = poly(last_12['days'].iloc[-1])
    offset = last_actual_price - last_predicted_price

    last_date = last_12['date'].max()
    future_dates = pd.date_range(start=last_date, periods=months_ahead + 1, freq='ME')[1:]
    future_days = (future_dates - last_12['date'].min()).days
    forecast_prices = poly(future_days) + offset

    forecast = pd.DataFrame({
        'date': future_dates,
        'price_cny': forecast_prices.round(0).astype(int),
        'type': 'Forecast'
    })

    # Add the last historical point so the line connects smoothly.
    last_hist = df_sorted.iloc[-1][['date', 'price_cny']].to_dict()
    last_hist['type'] = 'Forecast'
    forecast = pd.concat([pd.DataFrame([last_hist]), forecast], ignore_index=True)

    return forecast

In [ ]:
regions = ['World', 'China', 'European Union (27)', 'United States']

df, ev = load_and_clean_data('lithium_prices_raw.csv', 'ev_sales_raw.csv')
merged = merge_data(df, ev, regions)
correlations = calculate_correlation(merged)
seasonality = calculate_seasonality(df)
forecast = train_forecast(df)

merged.head()

In [ ]:
df_sorted = df.sort_values('date')
historical = df_sorted[['date', 'price_cny']].copy()
historical['type'] = 'Historical'
combined = pd.concat([historical, forecast], ignore_index=True)

fig = px.line(
    combined,
    x='date',
    y='price_cny',
    color='type',
    title='Lithium Price Forecast (Next 6 Months)',
    labels={'price_cny': 'Price (CNY/t)', 'date': 'Date'}
)
fig.show()

In [ ]:
# Save all datasets for the dashboard
merged.to_csv('lithium_ev_merged.csv', index=False)
seasonality.to_csv('seasonality.csv', index=False)
forecast.to_csv('forecast.csv', index=False)

# Save correlation results so app.py doesn't recalculate them
corr_df = pd.DataFrame([
    {"region": k, "correlation": v}
    for k, v in correlations.items()
])
corr_df.to_csv('correlation.csv', index=False)

print("All files saved successfully")